# MNIST 手写数字识别 — 项目笔记

《人工智能导论》课程演示项目

---

## 目录

1. [项目介绍](#1--项目介绍)
2. [环境准备](#2--环境准备)
3. [MNIST 数据集探索](#3--mnist-数据集探索)
4. [模型架构详解](#4--模型架构详解)
5. [训练过程](#5--训练过程)
6. [模型评估](#6--模型评估)
7. [自己动手预测](#7--自己动手预测)
8. [总结与扩展](#8--总结与扩展)

---

## 1.  项目介绍

### 我们要做什么？

构建一个**卷积神经网络（CNN）**，识别手写数字 0-9。这是深度学习领域的 "Hello World" 项目。

### 为什么是 MNIST？

- **简单**：28×28 灰度图，10 类分类，数据量适中
- **经典**：几乎所有深度学习教程都从它开始
- **有代表性**：包含完整的图像分类流程

### 整体流程

```
数据准备 → 模型定义 → 训练 → 评估 → 部署（Web 演示）
```

本笔记会带你走完前四个步骤，最终用 Web 界面（Gradio）实时感受模型的预测能力。

---

## 2.  环境准备

In [ ]:
# 检查 PyTorch 和 CUDA / MPS 可用性
import torch
import torchvision

print(f"PyTorch 版本: {torch.__version__}")
print(f"TorchVision 版本: {torchvision.__version__}")

if torch.cuda.is_available():
    print("  CUDA 可用 ✅ (NVIDIA GPU)")
elif torch.backends.mps.is_available():
    print("  MPS 可用 ✅ (Apple Silicon GPU)")
else:
    print("  使用 CPU 💻")

# 确保其他依赖可用
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: 已导入")
print(f"Pillow: {Image.__version__}")

> **提示**：如果你没有安装 `matplotlib`，可以运行 `pip install matplotlib`。
>
> 另外建议安装 `ipywidgets` 以获得更好的交互体验：`pip install ipywidgets`

---

## 3.  MNIST 数据集探索

MNIST（Modified National Institute of Standards and Technology）包含 60,000 张训练图片和 10,000 张测试图片。
每一张都是 **28×28 像素** 的灰度图，标签为 0-9 的手写数字。

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 不包含数据增强的变换（只是为了查看原始数据）
raw_transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.MNIST(
    root="./data", train=True, download=True, transform=raw_transform
)
test_dataset = datasets.MNIST(
    root="./data", train=False, download=True, transform=raw_transform
)

print(f"训练集大小: {len(train_dataset)}")
print(f"测试集大小: {len(test_dataset)}")
print(f"图片尺寸: {train_dataset[0][0].shape}")  # (1, 28, 28)
print(f"像素范围: [{train_dataset[0][0].min().item():.1f}, {train_dataset[0][0].max().item():.1f}]")
print(f"类别数: 10 (0-9)")

In [ ]:
# 可视化：展示训练集中每个类别的样本
%matplotlib inline

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('MNIST 数据集示例（每个类别一张）', fontsize=16, y=1.02)

seen = set()
row = 0
for img, label in train_dataset:
    if label not in seen:
        ax = axes[row // 5][row % 5]
        ax.imshow(img.squeeze(), cmap='gray')
        ax.set_title(f'标签: {label}', fontsize=14)
        ax.axis('off')
        seen.add(label)
        row += 1
    if len(seen) == 10:
        break

plt.tight_layout()
plt.show()

In [ ]:
# 类别分布统计
from collections import Counter

train_labels = [train_dataset[i][1] for i in range(len(train_dataset))]
test_labels = [test_dataset[i][1] for i in range(len(test_dataset))]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

train_counts = Counter(train_labels)
test_counts = Counter(test_labels)

ax1.bar(train_counts.keys(), train_counts.values(), color='steelblue')
ax1.set_title('训练集类别分布')
ax1.set_xlabel('数字')
ax1.set_ylabel('样本数')
ax1.set_xticks(range(10))

ax2.bar(test_counts.keys(), test_counts.values(), color='coral')
ax2.set_title('测试集类别分布')
ax2.set_xlabel('数字')
ax2.set_ylabel('样本数')
ax2.set_xticks(range(10))

plt.tight_layout()
plt.show()

print(f"各类别样本均衡，每类约 {(len(train_dataset)/10):.0f} 张训练图片")
print(f"各类别样本均衡，每类约 {(len(test_dataset)/10):.0f} 张测试图片")

### 关键观察

1. **类别均衡**：0-9 各类别样本数相差不大，无需做类平衡处理
2. **灰度图**：单通道（1×28×28），像素值 0（黑）~ 1（白）
3. **预处理标准化**：MNIST 的均值为 0.1307，标准差为 0.3081（这是数据集的统计量）

---

## 4.  模型架构详解

我们使用一个简单的 **CNN（卷积神经网络）**。下面逐层分析。

In [ ]:
# 导入模型定义
from model import MNISTNet

model = MNISTNet()
print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")

In [ ]:
# 可视化每一层输出的尺寸变化
def visualize_model_flow(model):
    """记录每层输出的形状变化"""
    layers_info = []

    def hook_fn(name):
        def hook(module, input, output):
            layers_info.append((name, list(output.shape)))
        return hook

    # 注册 hook
    handles = []
    for name, layer in model.named_modules():
        if isinstance(layer, (torch.nn.Conv2d, torch.nn.Linear, torch.nn.Dropout)):
            handle = layer.register_forward_hook(hook_fn(name))
            handles.append(handle)

    # 前向传播
    dummy = torch.randn(1, 1, 28, 28)
    model(dummy)

    # 清理
    for h in handles:
        h.remove()

    return layers_info


flow = visualize_model_flow(model)

print(f"{'层名称':<20} {'输出形状':<20} {'参数量'}")
print('-' * 60)
for name, shape in flow:
    # 查找对应层的参数量
    layer = dict(model.named_modules()).get(name, None)
    if layer is not None:
        params = sum(p.numel() for p in layer.parameters())
        print(f"{name:<20} {str(shape):<20} {params:,}")

### 架构图解

```
输入 (1×28×28)
    │
    ▼
Conv1 (1→16, 3×3, padding=1)  →  输出: 16×28×28
    │
    ▼
ReLU + MaxPool(2×2)            →  输出: 16×14×14
    │
    ▼
Conv2 (16→32, 3×3, padding=1)  →  输出: 32×14×14
    │
    ▼
ReLU + MaxPool(2×2)            →  输出: 32×7×7
    │
    ▼
Flatten (32×7×7 = 1568)
    │
    ▼
FC1 (1568 → 128) + ReLU + Dropout(0.25)
    │
    ▼
FC2 (128 → 10)
    │
    ▼
Softmax → 0-9 类别的概率
```

### 关键概念

| 概念 | 作用 |
|---|---|
| **卷积层** | 用可学习的卷积核扫描图像，提取局部特征（边缘、纹理、笔画） |
| **ReLU** | 激活函数，引入非线性，让网络能学习复杂模式 |
| **MaxPool** | 下采样，缩小特征图尺寸，提取最显著特征，减少参数量 |
| **Dropout** | 训练时随机丢弃部分神经元，防止过拟合 |
| **全连接层** | 将提取到的特征映射到类别空间 |
| **Softmax** | 将原始分数（logits）转换为概率分布 |

In [ ]:
# 可视化卷积核
conv1_weights = model.conv1.weight.data.cpu()  # (16, 1, 3, 3)

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
fig.suptitle('Conv1 的 16 个卷积核（3×3）', fontsize=14)

for i, ax in enumerate(axes.flat):
    if i < conv1_weights.shape[0]:
        kernel = conv1_weights[i, 0]  # (3, 3)
        # 归一化到 [0,1] 以便显示
        vmin, vmax = kernel.min(), kernel.max()
        kernel_norm = (kernel - vmin) / (vmax - vmin) if vmax > vmin else kernel
        ax.imshow(kernel_norm, cmap='viridis', interpolation='nearest')
        ax.set_title(f'核 {i}')
    ax.axis('off')

plt.tight_layout()
plt.show()
print("提示：这些 3×3 的卷积核在训练过程中自动学习如何检测笔画边缘和纹理。")

---

## 5.  训练过程

我们从 `train.py` 中提取训练逻辑，在这里执行完整训练或加载已有模型。

如果之前已经跑过 `python train.py`，可以跳过训练直接加载已保存的模型。

In [ ]:
import os
import time
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# 超参数
BATCH_SIZE = 64
EPOCHS = 5
LEARNING_RATE = 0.001

# 设备
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("使用 NVIDIA GPU (CUDA) 训练")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("使用 Apple Silicon GPU (MPS) 训练")
else:
    device = torch.device("cpu")
    print("使用 CPU 训练")

# 数据增强（训练集）
train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=5),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

# 数据加载
train_dataset_aug = datasets.MNIST(
    root="./data", train=True, download=True, transform=train_transform
)
test_dataset_aug = datasets.MNIST(
    root="./data", train=False, download=True, transform=test_transform
)

train_loader = DataLoader(train_dataset_aug, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset_aug, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# 检查是否已有训练好的模型
MODEL_PATH = "mnist_model.pth"

if os.path.exists(MODEL_PATH):
    print(f"✅ 发现已保存的模型文件 {MODEL_PATH}")
    print("提示：跳过训练，直接加载模型。想重新训练的话，删除该文件再运行下面单元格。")
else:
    print("需要训练模型，运行下一个单元格...")

In [ ]:
# 训练模型（如果有需要）
if not os.path.exists(MODEL_PATH):
    model = MNISTNet().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}

    print(f"开始训练，共 {EPOCHS} 个 epoch...")
    print('=' * 60)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        start_time = time.time()

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

            if (batch_idx + 1) % 200 == 0:
                print(f"  Epoch {epoch}/{EPOCHS} | Batch {batch_idx+1}/{len(train_loader)} | "
                      f"Loss: {running_loss/(batch_idx+1):.4f} | Acc: {100.*correct/total:.1f}%")

        # 每个 epoch 后的评估
        model.eval()
        test_correct = 0
        test_total = 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                _, predicted = output.max(1)
                test_total += target.size(0)
                test_correct += predicted.eq(target).sum().item()

        train_acc = 100.0 * correct / total
        test_acc = 100.0 * test_correct / test_total
        elapsed = time.time() - start_time

        history['train_loss'].append(running_loss / len(train_loader))
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)

        print(f"✅ Epoch {epoch}/{EPOCHS} 完成 | 耗时 {elapsed:.1f}s | "
              f"训练 Acc: {train_acc:.1f}% | 测试 Acc: {test_acc:.1f}%")
        print('=' * 60)

    # 保存模型
    torch.save(model.state_dict(), MODEL_PATH)
    print(f"\n💾 模型已保存到 {MODEL_PATH}")
    print(f"🎉 最终测试准确率: {test_acc:.1f}%")
else:
    print("⏭️ 已有训练好的模型，跳过训练。")

In [ ]:
# 加载模型
model = MNISTNet()
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu", weights_only=True))
model.eval()
print("✅ 模型已加载到 CPU（推理模式）")
print(f"   模型文件大小: {os.path.getsize(MODEL_PATH) / 1024:.1f} KB")

---

## 6.  模型评估

在测试集上评估模型，并查看一些有趣的错误案例。

In [ ]:
# 在测试集上计算总体准确率
test_correct = 0
test_total = 0
all_preds = []
all_targets = []

with torch.no_grad():
    for data, target in test_loader:
        output = model(data)
        _, predicted = output.max(1)
        test_total += target.size(0)
        test_correct += predicted.eq(target).sum().item()
        all_preds.extend(predicted.numpy())
        all_targets.extend(target.numpy())

test_acc = 100.0 * test_correct / test_total
print(f"测试集准确率: {test_acc:.2f}%")
print(f"正确: {test_correct} / {test_total}")

In [ ]:
# 混淆矩阵
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(all_targets, all_preds)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=range(10))
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title(f'MNIST 混淆矩阵（准确率 {test_acc:.1f}%）', fontsize=14)
plt.show()

# 计算每类准确率
print("各类别准确率:")
for i in range(10):
    total_i = np.sum(np.array(all_targets) == i)
    correct_i = np.sum((np.array(all_targets) == i) & (np.array(all_preds) == i))
    print(f"  数字 {i}: {correct_i}/{total_i} = {100*correct_i/total_i:.1f}%")

In [ ]:
# 查看错误分类的样例
misclassified_indices = [
    i for i in range(len(all_targets)) if all_preds[i] != all_targets[i]
]

print(f"在测试集中共有 {len(misclassified_indices)} 个错误分类 (占 {100*len(misclassified_indices)/len(all_targets):.2f}%)")

# 展示前 20 个错误
fig, axes = plt.subplots(4, 5, figsize=(12, 10))
fig.suptitle(f'错误分类示例（共 {len(misclassified_indices)} 个）', fontsize=14, y=1.01)

for i, idx in enumerate(misclassified_indices[:20]):
    ax = axes[i // 5][i % 5]
    # 获取原始图片
    img, _ = test_dataset[idx]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'真实: {all_targets[idx]} → 预测: {all_preds[idx]}', fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.show()
print("观察：这些错误案例中，很多对于人类来说也比较模糊。")

### 错误分析

常见的错误类型：
- **形状相似**：4↔9, 3↔8, 7↔1 这类易混淆
- **书写潦草**：笔画不规范的写法
- **边缘截断**：数字超出画布边界

如果把这些错误案例加入到训练集中重新训练，模型可能会变得更好——这就是 **主动学习（Active Learning）** 的思路。

---

## 7.  自己动手预测

在下面手写一个数字，试试模型的预测效果！

> **注意**：如果你在 Jupyter 环境中运行，下面的交互式控件需要 `ipywidgets` 支持。
> 如果无法显示画板，可以改用代码方式：直接修改下方 `image_array` 单元格中手动构造的 28×28 数组。

In [ ]:
# 从测试集中随机选一张图片进行预测
import random

idx = random.randint(0, len(test_dataset) - 1)
img_tensor, true_label = test_dataset[idx]

with torch.no_grad():
    output = model(img_tensor.unsqueeze(0))  # 加 batch 维度
    probs = torch.nn.functional.softmax(output, dim=1)[0]

pred_label = torch.argmax(probs).item()
confidence = probs[pred_label].item()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# 图片
ax1.imshow(img_tensor.squeeze(), cmap='gray')
ax1.set_title(f'测试集图片\n真实值: {true_label}', fontsize=13)
ax1.axis('off')

# 概率柱状图
colors = ['coral' if i == pred_label else 'steelblue' for i in range(10)]
ax2.bar(range(10), probs.numpy(), color=colors)
ax2.set_title(f'预测结果: {pred_label} (置信度: {confidence:.1%})', fontsize=13)
ax2.set_xlabel('数字')
ax2.set_ylabel('概率')
ax2.set_xticks(range(10))

plt.tight_layout()
plt.show()

if pred_label == true_label:
    print(f"✅ 预测正确！模型认为数字 {pred_label} 的概率为 {confidence:.1%}")
else:
    print(f"❌ 预测错误！真实值为 {true_label}，模型预测为 {pred_label}（概率 {confidence:.1%}）")

In [ ]:
# 批量预览：展示多个随机测试样本的预测结果
fig, axes = plt.subplots(3, 5, figsize=(14, 8))
fig.suptitle('随机测试样本 —— 模型预测 vs 真实标签', fontsize=15, y=1.01)

indices = random.sample(range(len(test_dataset)), 15)

for ax, idx in zip(axes.flat, indices):
    img_tensor, true_label = test_dataset[idx]

    with torch.no_grad():
        output = model(img_tensor.unsqueeze(0))
        probs = torch.nn.functional.softmax(output, dim=1)[0]

    pred_label = torch.argmax(probs).item()
    correct = pred_label == true_label

    ax.imshow(img_tensor.squeeze(), cmap='gray')
    color = 'green' if correct else 'red'
    ax.set_title(f'真实:{true_label} → 预测:{pred_label}\n置信度:{probs[pred_label]:.2%}',
                 color=color, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

---

## 8.  总结与扩展

### 我们学到了什么？

| 知识点 | 对应代码 |
|---|---|
| CNN 基本结构（卷积 → 池化 → 全连接） | `model.py` |
| 数据加载与增强 | `train.py` → `DataLoader`, `transforms` |
| 训练循环（前向 → 计算损失 → 反向传播 → 更新） | `train.py` → `train()` |
| 模型评估与混淆矩阵 | 本笔记第 6 节 |
| 模型部署（Web 界面） | `app.py` → Gradio |

### 为什么准确率能到 99%？

1. **MNIST 比较简单**：28×28 灰度图，类别少，笔画清晰
2. **数据增强**：随机旋转、平移、缩放让模型见过更多变化
3. **Dropout 防过拟合**：训练时随机丢弃神经元，增强泛化能力

### 扩展方向

如果你觉得这个项目不过瘾，可以尝试：

1. **Fashion-MNIST**：更难的替代数据集，同样的 28×28 尺寸和 10 分类
2. **更深/更宽的网络**：增加卷积层数或通道数 → 但要注意过拟合
3. **超参数搜索**：试试不同的学习率、batch size、dropout ratio
4. **部署到 Hugging Face Spaces**：生成的 Gradio 应用可以免费托管
5. **可视化中间特征**：用 hook 查看卷积层输出的 feature map，理解 CNN "看到了什么"

---

*Happy Coding! 🚀*